In [9]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

df_fh_rsl = pd.read_csv(PROCESSED_DIR / "fh_rsl_clean.csv")
df_sanity = pd.read_csv(PROCESSED_DIR / "sanity_clean.csv")
df_rxlev = pd.read_csv(PROCESSED_DIR / "rxlev_clean.csv")
df_rtwp = pd.read_csv(PROCESSED_DIR / "rtwp_clean.csv")
df_atoll = pd.read_csv(PROCESSED_DIR / "atoll_links_clean.csv")

for name, df in [("fh_rsl", df_fh_rsl), ("sanity", df_sanity), ("rxlev", df_rxlev),
                  ("rtwp", df_rtwp), ("atoll", df_atoll)]:
    print(name, df.shape)

fh_rsl (1292, 9)
sanity (6038, 17)
rxlev (6016, 9)
rtwp (295, 8)
atoll (3795, 37)


sanity and rxlev — RSL range

In [10]:
df_sanity["rsl_range"] = df_sanity["RSL (Max)"] - df_sanity["RSL (Min)"]
df_rxlev["rsl_range"] = df_rxlev["Max RSL"] - df_rxlev["Min RSL"]

print(df_sanity["rsl_range"].describe())
print(df_rxlev["rsl_range"].describe())

count    4612.000000
mean       35.786882
std        25.762412
min         0.100000
25%         9.700000
50%        32.900000
75%        62.100000
max        86.500000
Name: rsl_range, dtype: float64
count    4614.000000
mean       35.773364
std        25.765061
min         0.100000
25%         9.700000
50%        32.850000
75%        62.100000
max        86.500000
Name: rsl_range, dtype: float64


rtwp — antenna imbalance magnitude + RTWP spread

In [11]:
df_rtwp["antenna_imbalance"] = (df_rtwp["vs_mean_rtwp_ant0"] - df_rtwp["vs_mean_rtwp_ant1"]).abs()
df_rtwp["rtwp_range"] = df_rtwp["max_rtwp"] - df_rtwp["min_rtwp"]

print(df_rtwp[["antenna_imbalance", "rtwp_range"]].describe())

       antenna_imbalance  rtwp_range
count         295.000000  295.000000
mean            2.830785   13.367720
std             2.772705    6.404732
min             0.011300    1.067400
25%             0.693300    9.294350
50%             1.918200   12.597500
75%             4.349800   15.657850
max            12.747800   36.459500


atoll_links — End A vs End B asymmetry features

In [12]:
end_pairs = [
    "antenna_gain_db", "tx_power_dbm", "threshold_dbm",
    "main_rx_level_dbm", "main_flat_fade_margin_db",
]

for field in end_pairs:
    col_a = f"{field}_end_a"
    col_b = f"{field}_end_b"
    df_atoll[f"{field}_asymmetry"] = (df_atoll[col_a] - df_atoll[col_b]).abs()

# fade margin is a key telecom health indicator: how much margin above threshold
df_atoll["fade_margin_min"] = df_atoll[["main_flat_fade_margin_db_end_a", "main_flat_fade_margin_db_end_b"]].min(axis=1)

asymmetry_cols = [f"{f}_asymmetry" for f in end_pairs] + ["fade_margin_min"]
print(df_atoll[asymmetry_cols].describe())

       antenna_gain_db_asymmetry  tx_power_dbm_asymmetry  \
count                3795.000000             3795.000000   
mean                    0.207589                0.001054   
std                     1.007850                0.045907   
min                     0.000000                0.000000   
25%                     0.000000                0.000000   
50%                     0.000000                0.000000   
75%                     0.000000                0.000000   
max                     8.000000                2.000000   

       threshold_dbm_asymmetry  main_rx_level_dbm_asymmetry  \
count                   3795.0                  3795.000000   
mean                       0.0                     0.416021   
std                        0.0                     0.437838   
min                        0.0                     0.000000   
25%                        0.0                     0.300000   
50%                        0.0                     0.380000   
75%               

In [13]:
df_atoll = df_atoll.drop(columns=["tx_power_dbm_asymmetry", "threshold_dbm_asymmetry"])
df_atoll.to_csv(PROCESSED_DIR / "atoll_links_features.csv", index=False)
print(df_atoll.shape)

(3795, 41)


In [14]:
print(df_fh_rsl.groupby("Status")["RSL DIFF"].describe())
print()
print(df_sanity.groupby("Sanity")["rsl_range"].describe())

                     count       mean        std    min      25%     50%  \
Status                                                                     
Lien_OK              826.0  -1.170508   4.302073 -21.35  -3.5975  -0.830   
Lien_dépointé_(<10)  238.0   7.345630   1.399725   5.02   6.1925   7.255   
Lien_dépointé_(>10)  198.0  22.042828  14.852904  10.08  12.6200  17.025   

                         75%    max  
Status                               
Lien_OK               2.1975   4.99  
Lien_dépointé_(<10)   8.4750   9.99  
Lien_dépointé_(>10)  22.9600  94.31  

             count       mean        std  min     25%   50%   75%   max
Sanity                                                                 
CURATIVE    1428.0  61.909314   9.742496  5.6  57.875  63.8  68.4  86.5
OK          1103.0   8.939710   4.779767  0.5   5.000   8.0  12.1  28.0
PREVENTIVE  2081.0  32.091350  23.015846  0.1  10.900  25.2  54.8  82.1


Save all feature-engineered datasets

In [15]:
df_sanity.to_csv(PROCESSED_DIR / "sanity_features.csv", index=False)
df_rxlev.to_csv(PROCESSED_DIR / "rxlev_features.csv", index=False)
df_fh_rsl.to_csv(PROCESSED_DIR / "fh_rsl_features.csv", index=False)
df_rtwp.to_csv(PROCESSED_DIR / "rtwp_features.csv", index=False)
df_atoll.to_csv(PROCESSED_DIR / "atoll_links_features.csv", index=False)

for name in ["sanity_features", "rxlev_features", "fh_rsl_features", "rtwp_features", "atoll_links_features"]:
    p = PROCESSED_DIR / f"{name}.csv"
    print(name, "->", p.exists(), p.stat().st_size, "bytes")

sanity_features -> True 617548 bytes
rxlev_features -> True 506358 bytes
fh_rsl_features -> True 147900 bytes
rtwp_features -> True 31366 bytes
atoll_links_features -> True 1196802 bytes


# 03 — Feature Engineering

**Goal:** Derive telecom-meaningful features from the cleaned datasets, and validate
that they actually separate the target classes before modeling.

**Inputs:** `data/processed/*_clean.csv`

**Features added:**
- `rsl_range` (sanity, rxlev) — RSL Max − RSL Min, signal instability indicator
- `antenna_imbalance`, `rtwp_range` (rtwp)
- End A/B asymmetry features (atoll_links): antenna gain, main Rx level, fade margin
  (dropped tx_power and threshold asymmetry — constant, no signal)

**Validation:**
- `RSL DIFF` cleanly separates `Status` classes (Lien_OK mean -1.2 → dépointé(>10) mean 22.0)
- `rsl_range` cleanly separates `Sanity` classes (OK mean 8.9 → CURATIVE mean 61.9)

**Outputs:** `data/processed/sanity_features.csv`, `rxlev_features.csv`, `fh_rsl_features.csv`,
`rtwp_features.csv`, `atoll_links_features.csv`